# NES-VMC（Natural Excited State Variational Monte Carlo）算法实现

## 关于 Model 输出的重要说明

**NetKet 约定**：在 NetKet 中，神经网络 model 的输出默认是 **log 取值**，即输出 $\log \psi(x)$ 而不是 $\psi(x)$。

这意味着：
- 局部能量计算：$\frac{\psi(\eta)}{\psi(\sigma)} = \exp(\log \psi(\eta) - \log \psi(\sigma))$
- 采样：基于 $|\psi|^2 = \exp(2\text{Re}(\log \psi))$

在 NES-VMC 中，我们需要：
- 行列式计算：$\det(\Psi)$ 其中 $\Psi_{ij} = \psi_j(x^i)$
- 由于 $\det(\exp A) = \exp(\text{Tr}(A))$，我们可以使用对数形式

$$\nabla_{\theta} \mathcal{L} = 2 \mathbb{E}_{\mathbf{x} \sim |\Psi|^2} \left[ \mathrm{Tr}\left( E_L(\mathbf{x}) - \bar{E}_L \right) \nabla_{\theta} \log \Psi(\mathbf{x})^* \right]$$

## 1. 环境配置与 FCI 基准

In [2]:
# ===================== 环境配置 =====================
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from jax import flatten_util
import matplotlib.pyplot as plt
from tqdm import tqdm
from jax import vmap,jit,grad
from NES_VMC import create_machine,hi_ext,hi,\
sampler,SingleStateAnsatz,NESTotalAnsatz,ha,compute_loss_and_grad,E_fcis,compute_qgt



/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


In [18]:
K=2
lr=0.05           # 自然梯度专用学习率
n_iter = 500
chain_length = 150  # 足够的采样长度
burn_in = 100      # 【关键】采样烧穿步（消除初始偏差）

# 初始化模型
rngs = nnx.Rngs(21)
total_ansatz = NESTotalAnsatz(4, K, 8, rngs=rngs)
machine, graphdef, params = create_machine(total_ansatz)

# 初始化采样器状态
sampler_state = sampler.init_state(machine, params, seed=12)

# 优化器：Adam 适配自然梯度（秒杀 SGD）
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# 训练循环
E_L_history = []
loss_history = []

print(f'FCI 基准：基态={E_fcis[0]:.8f} | 第一激发态={E_fcis[1]:.8f}')
print(f'FCI 总能量（理论损失）：{E_fcis[0]+E_fcis[1]:.8f}')

for step in range(n_iter):
    # ===================== 【核心修复】采样器重置 + 烧穿 =====================
    sampler_state = sampler.reset(machine, params, sampler_state)
    # 烧穿：丢弃初始无效样本（VMC 必做！）
    _, sampler_state = sampler.sample(machine, params, state=sampler_state, chain_length=burn_in)
    
    # 1. 有效采样
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, chain_length=chain_length
    )    
    
    # 2. 计算损失和梯度
    loss, grads, E_L_avg = compute_loss_and_grad(graphdef, params, samples, ha, K)
    
    # ===================== 物理约束修复 =====================
    loss = jnp.real(loss)                # 抹去虚部（实数体系）
    E_L_avg = jnp.real(E_L_avg)          # 能量矩阵实数化
    E_L_avg = (E_L_avg + E_L_avg.T) / 2  # 厄米化（NES 强制要求）
    
    # 3. 自然梯度计算（稳定版）
    grad_flat, grad_unravel_fn = flatten_util.ravel_pytree(grads)
    qgt_mat, _ = compute_qgt(machine, params, samples, 0.001) 
    natural_grad_flat = jnp.linalg.solve(qgt_mat, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad_flat)

    # 4. 参数更新（用自然梯度）
    updates, opt_state = optimizer.update(natural_grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 记录
    loss_history.append(loss)
    E_L_history.append(E_L_avg)
    
    if step % 10 == 0:
        eig_vals = jnp.linalg.eigvalsh(E_L_avg)
        eig_vals = jnp.sort(eig_vals)  # 强制排序：基态 < 激发态
        print(f"Step {step:4d} | Loss: {loss:.8f} | 能量: {eig_vals[0]:.8f}  {eig_vals[1]:.8f}")

FCI 基准：基态=-1.01546825 | 第一激发态=-0.87542794
FCI 总能量（理论损失）：-1.89089619
Step    0 | Loss: -0.88327210 | 能量: -1.17702145  0.29374935
Step   10 | Loss: -1.07657920 | 能量: -0.82977303  -0.24680617
Step   20 | Loss: -1.45837824 | 能量: -0.95780154  -0.50057669
Step   30 | Loss: -1.45978602 | 能量: -0.89579426  -0.56399176
Step   40 | Loss: -1.50853347 | 能量: -0.90049332  -0.60804015
Step   50 | Loss: -1.30584993 | 能量: -0.71363603  -0.59221390
Step   60 | Loss: -1.33013411 | 能量: -0.82774172  -0.50239239
Step   70 | Loss: -1.32426585 | 能量: -0.76505210  -0.55921375
Step   80 | Loss: -1.01253855 | 能量: -0.75140628  -0.26113226
Step   90 | Loss: -1.05590108 | 能量: -0.66678657  -0.38911451
Step  100 | Loss: -1.08882154 | 能量: -0.81337364  -0.27544790
Step  110 | Loss: -1.12702216 | 能量: -0.71647307  -0.41054908
Step  120 | Loss: -1.11476327 | 能量: -1.02232105  -0.09244222
Step  130 | Loss: -1.19857933 | 能量: -0.85409883  -0.34448050
Step  140 | Loss: -1.19784712 | 能量: -0.89708877  -0.30075835
Step  150 | Loss: 

KeyboardInterrupt: 

In [4]:
rngs = nnx.Rngs(21)
K=2
total_ansatz = NESTotalAnsatz(4, K, 8, rngs=rngs)
single_ansatz = SingleStateAnsatz(4, 8, rngs=rngs)